# Customer Feature Embedding Inspection

This notebook builds and inspects the engineered customer feature matrix before any compression.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
SRC_DIR = ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from embeddings.customer_features import (
    CustomerFeatureConfig,
    build_customer_feature_matrix,
    read_lumen_csv,
    schema_report,
)

DATA_PATH = Path(r'C:\\Users\\lovro\\Desktop\\hackatoni\\LUMEN_DS_processed.csv')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)


In [ ]:
raw_df = read_lumen_csv(DATA_PATH)
print(raw_df.shape)
display(raw_df.head())
display(schema_report(raw_df))


## Feature Blocks

- aggregate transaction, quantity, price, cost, margin, and fulfillment features
- full-share blocks for semantic columns like `Product group` and `Product family`
- bucketed product-group features ranked by global sales count in groups of 5
- bucket magnitude features like `bucket_1_5_total_count` and `bucket_1_5_total_revenue`
- family hierarchy features: family totals plus within-family group shares


In [ ]:
config = CustomerFeatureConfig(bucket_size=5)
customer_features, metadata = build_customer_feature_matrix(raw_df, config, return_metadata=True)
print(customer_features.shape)
display(customer_features.head())
display(metadata['bucket_map'].head(20))


In [ ]:
feature_groups = metadata['feature_groups']
for group_name, columns in feature_groups.items():
    print(group_name, len(columns))

display(pd.Series(feature_groups['bucket_magnitudes'], name='bucket_magnitude_columns'))
display(pd.Series(feature_groups['family_hierarchy'][:40], name='family_hierarchy_columns'))


In [ ]:
sample_customers = customer_features.index[:5].tolist()
for customer_id in sample_customers:
    print(f'customer {customer_id}')
    non_zero_bucket_shares = customer_features.loc[customer_id, feature_groups['bucket_shares']]
    non_zero_bucket_shares = non_zero_bucket_shares[non_zero_bucket_shares > 0].sort_values(ascending=False)
    display(non_zero_bucket_shares.head(20).to_frame('bucket_local_share'))
    bucket_totals = customer_features.loc[customer_id, feature_groups['bucket_magnitudes']]
    bucket_totals = bucket_totals[bucket_totals > 0].sort_values(ascending=False)
    display(bucket_totals.head(20).to_frame('bucket_magnitude'))
    family_values = customer_features.loc[customer_id, feature_groups['family_hierarchy']]
    family_values = family_values[family_values > 0].sort_values(ascending=False)
    display(family_values.head(20).to_frame('family_hierarchy_value'))


In [ ]:
null_ratio = customer_features.isna().mean().sort_values(ascending=False)
display(null_ratio.head(30).to_frame('null_ratio'))

plt.figure(figsize=(10, 4))
metadata['bucket_map']['bucket_id'].value_counts().sort_index().plot(kind='bar')
plt.title('Number of Product Groups per Global-Rank Bucket')
plt.xlabel('Bucket ID')
plt.ylabel('Number of Groups')
plt.tight_layout()
plt.show()
